# 🚀 NVIDIA GR00T-N1.7 VLA PolicyServer trên Google Colab

Notebook chạy model **GR00T N1.7 thật** trên GPU Colab và nối với mô phỏng MuJoCo trên máy local qua đường hầm TCP Pinggy. Dùng được cho checkpoint base (`nvidia/GR00T-N1.7-3B`, embodiment `REAL_G1`) và checkpoint fine-tune từ `colab_gr00t_finetune.ipynb` (embodiment `new_embodiment`).

### Yêu cầu
* **GPU:** nên dùng **L4** hoặc **A100**. T4 vẫn chạy được (server tự tắt flash-attn), nhưng bfloat16 trên T4 là giả lập nên suy luận chậm; khi đó ở máy local tăng `--server-timeout-ms`.
* **HuggingFace:** backbone `nvidia/Cosmos-Reason2-2B` là repo **gated**. Tài khoản của bạn phải được cấp quyền trên trang model, rồi thêm token vào **Colab Secrets** (biểu tượng 🔑 bên trái) với tên `HF_TOKEN` và bật *Notebook access*.

### Các bước
1. **Runtime → Change runtime type → L4 GPU** (hoặc A100).
2. Chạy lần lượt từ Bước 1 đến Bước 6. Cell `%%writefile` (Bước 4) ghi file server. Ở Bước 5 chọn checkpoint; server **chạy nền**.
3. Copy lệnh in ra ở Bước 6 sang terminal máy local.
4. Tunnel Pinggy miễn phí hết hạn sau khoảng 60 phút: chạy lại Bước 6 để lấy host/port mới, không cần nạp lại model.
   Bước 7 xem trạng thái và log server; Bước 8 dừng server (phải đặt `STOP_SERVER = True`).

Với checkpoint base, model chưa từng thấy cảnh sim này nên robot sẽ cử động nhưng gần như chắc chắn không gắp được lon.

Isaac-GR00T được ghim ở commit `51d4c89`.

In [ ]:
# [BƯỚC 1] Kiểm tra GPU
!nvidia-smi

In [ ]:
%%bash
# [BƯỚC 2] Cài Isaac-GR00T vào môi trường riêng (.venv, Python 3.12) bằng uv (~5-10 phút).
# set -e: dừng ngay ở lệnh lỗi đầu tiên để thấy rõ nguyên nhân (lệnh "!" của Colab không tự dừng khi lỗi).
set -euxo pipefail
cd /content
[ -d Isaac-GR00T/.git ] || GIT_LFS_SKIP_SMUDGE=1 git clone https://github.com/NVIDIA/Isaac-GR00T
cd Isaac-GR00T
git checkout 51d4c89f72fda44cbf77285c6a8114b52676b8a1
# Clone bỏ qua file Git LFS (video, dataset demo) cho nhẹ, nhưng uv cần wheel LFS trong
# scripts/deployment/dgpu/wheels để giải lock (kể cả wheel aarch64 không dùng trên Colab): tải riêng thư mục này.
command -v git-lfs || (apt-get -qq update && apt-get -qq install -y git-lfs)
git lfs install --local
git lfs pull --include="scripts/deployment/dgpu/wheels/*"
# Trình cài uv chọn /usr/local/bin hoặc ~/.local/bin tùy môi trường: gọi uv qua PATH, không dùng đường dẫn cứng
export PATH="$HOME/.local/bin:/usr/local/bin:$PATH"
command -v uv || curl -LsSf https://astral.sh/uv/install.sh | sh
# Venv luôn nằm ở /content/Isaac-GR00T/.venv (bỏ qua biến môi trường nếu có)
unset UV_PROJECT_ENVIRONMENT VIRTUAL_ENV
uv sync --python 3.12
.venv/bin/python -c "import gr00t; print('GR00T installed successfully')"

In [ ]:
# [BƯỚC 3] Nạp HuggingFace token từ Colab Secrets (không dán token trực tiếp vào notebook)
# và kiểm tra ngay quyền tải backbone gated nvidia/Cosmos-Reason2-2B.
# Chạy lại cell này mỗi khi runtime khởi động lại.
import os
from huggingface_hub import hf_hub_download, whoami
from huggingface_hub.utils import GatedRepoError

try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception as e:
    print(f"Không đọc được secret HF_TOKEN ({type(e).__name__}); nhập token thủ công.")
    from getpass import getpass
    token = getpass("HuggingFace token: ")
token = (token or "").strip()
assert token, "HF_TOKEN rỗng: thêm secret tên HF_TOKEN (biểu tượng 🔑) và bật Notebook access."
os.environ["HF_TOKEN"] = token

try:
    user = whoami(token=token)
except Exception as e:
    raise RuntimeError(
        "Token không hợp lệ hoặc đã bị thu hồi. Tạo token mới tại https://huggingface.co/settings/tokens "
        "rồi cập nhật secret HF_TOKEN."
    ) from e
print(f"Token hợp lệ, tài khoản: {user['name']}")

try:
    hf_hub_download("nvidia/Cosmos-Reason2-2B", "config.json", token=token)
except GatedRepoError as e:
    raise RuntimeError(
        f"Tài khoản {user['name']} chưa tải được nvidia/Cosmos-Reason2-2B. Kiểm tra:\n"
        "  1. Đã bấm xin quyền và được duyệt tại https://huggingface.co/nvidia/Cosmos-Reason2-2B "
        "(đăng nhập đúng tài khoản trên).\n"
        "  2. Nếu là token fine-grained: bật quyền 'Read access to contents of all public gated repos "
        "you can access', hoặc dùng token loại Read."
    ) from e
print("✅ Có quyền tải nvidia/Cosmos-Reason2-2B.")

In [ ]:
%%writefile /content/colab_gr00t_server.py
"""
Script colab_gr00t_server.py: ZMQ Policy Server cho NVIDIA GR00T N1.7 trên Google Colab GPU.
Nhận ảnh camera ego_view + câu lệnh ngôn ngữ + trạng thái robot từ máy local và trả về action chunk.

Hai chế độ (--policy):
  - gr00t (mặc định): model GR00T N1.7 thật qua API Gr00tPolicy của Isaac-GR00T. Checkpoint base dùng embodiment
    REAL_G1 (real_g1_relative_eef_relative_joints); checkpoint fine-tune bằng finetune/g1_can_sort_config.py dùng
    --embodiment-tag new_embodiment với cùng layout. Trả về góc khớp tay/bàn tay/eo tuyệt đối, action_space "real_g1".
    Cần chạy bằng Python của môi trường Isaac-GR00T và HF_TOKEN có quyền truy cập nvidia/Cosmos-Reason2-2B.
  - scripted: quỹ đạo phân loại lon mẫu dạng motion token [40, 78], action_space "sonic_tokens".
"""

import argparse
import io
import os
import traceback
import pickle
import sys
import time
import numpy as np

try:
    import torch
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False

try:
    import zmq
    HAS_ZMQ = True
except ImportError:
    HAS_ZMQ = False


# Quỹ đạo phân loại lon mẫu: bản sao của bridge/gr00t_policy_client.py (server chạy độc lập trên Colab,
# không import được package bridge). Khi đổi waypoint, lịch trình hay tư thế bàn tay phải sửa cả hai nơi.
# Waypoint cho tay phải: [waist_yaw, waist_pitch, s_pitch, s_roll, s_yaw, elbow, wr_roll, wr_pitch, wr_yaw]
WAYPOINTS = {
    "ready":   [0.0,  0.000,  0.000,  0.000,  0.000,  0.000,  0.000,  0.000,  0.000],
    "pre":     [0.0,  0.303, -0.037, -0.492,  0.001,  0.235, -0.243, -0.473, -0.878],
    "grasp":   [0.0,  0.834, -0.304, -0.601, -0.091,  0.342, -0.372, -0.495, -1.233],
    "lift":    [0.0, -0.314, -0.143, -0.449, -0.177, -0.195, -0.308,  0.293, -0.994],
    "carry":   [0.0,  0.206, -0.374, -0.150,  0.194,  0.105, -0.253,  0.231, -1.512],
    "place":   [0.0,  1.000, -0.517, -0.270,  0.244,  0.430, -0.261, -0.236, -1.748],
    "retreat": [0.0, -0.424, -0.262,  0.027,  0.176, -0.125, -0.208,  0.617, -1.192],
}
SCHEDULE = [
    (0.00, "ready", 0.0), (0.18, "pre", 0.0), (0.30, "grasp", 0.0), (0.38, "grasp", 1.0),
    (0.50, "lift", 1.0), (0.64, "carry", 1.0), (0.74, "place", 1.0), (0.80, "place", 0.0),
    (0.88, "retreat", 0.0), (1.00, "ready", 0.0),
]
CYCLE_STEPS = 350
PEPSI_MIRROR = np.array([-1, 1, 1, 1, 1, 1, -1, 1, -1], dtype=np.float32)
RIGHT_ARM_TOKEN_IDX = [12, 14, 22, 23, 24, 25, 26, 27, 28]
LEFT_ARM_TOKEN_IDX = [12, 14, 15, 16, 17, 18, 19, 20, 21]
RIGHT_HAND_CLOSED = np.array([0.0, -0.5, -0.6, 0.7, 0.7, 0.7, 0.7], dtype=np.float32)
LEFT_HAND_CLOSED = -RIGHT_HAND_CLOSED

_KEY_PHASES = np.array([k[0] for k in SCHEDULE], dtype=np.float32)
_KEY_POSES = np.array([WAYPOINTS[k[1]] for k in SCHEDULE], dtype=np.float32)
_KEY_GRIPS = np.array([k[2] for k in SCHEDULE], dtype=np.float32)


def scripted_sorting_chunk(task_description: str, progress: float, horizon: int, cycle_steps: int) -> np.ndarray:
    """Sinh action chunk [horizon, 78] của quỹ đạo phân loại lon mẫu bắt đầu tại tiến trình `progress`.

    progress chạy từ 0 đến 1 trong một lần gắp-đặt; hết chu kỳ thì robot giữ tư thế ready (nhiệm vụ đã xong)
    thay vì lặp lại và hạ tay trúng lon đã nằm trong khay.
    """
    desc = task_description.lower()
    is_coke = "coca" in desc or "coke" in desc or "red" in desc

    phase = np.minimum(progress + np.arange(horizon, dtype=np.float32) / cycle_steps, 1.0)
    seg = np.clip(np.searchsorted(_KEY_PHASES, phase, side="right") - 1, 0, len(SCHEDULE) - 2)
    t = np.clip((phase - _KEY_PHASES[seg]) / (_KEY_PHASES[seg + 1] - _KEY_PHASES[seg]), 0.0, 1.0)
    w = (t * t * (3.0 - 2.0 * t))[:, None]
    pose = (1.0 - w) * _KEY_POSES[seg] + w * _KEY_POSES[seg + 1]
    grip = (1.0 - w[:, 0]) * _KEY_GRIPS[seg] + w[:, 0] * _KEY_GRIPS[seg + 1]

    action = np.zeros((horizon, 78), dtype=np.float32)
    if is_coke:
        action[:, RIGHT_ARM_TOKEN_IDX] = pose
        action[:, 71:78] = grip[:, None] * RIGHT_HAND_CLOSED
    else:
        action[:, LEFT_ARM_TOKEN_IDX] = pose * PEPSI_MIRROR
        action[:, 64:71] = grip[:, None] * LEFT_HAND_CLOSED
    return action


# Embodiment của checkpoint base GR00T-N1.7-3B ứng với robot G1 (xem bridge/real_g1_adapter.py ở máy local)
DEFAULT_EMBODIMENT_TAG = "real_g1_relative_eef_relative_joints"


class ScriptedBackend:
    """Quỹ đạo phân loại lon mẫu, không dùng model."""

    action_space = "sonic_tokens"
    model_info = {"model_path": "scripted", "embodiment_tag": None}

    def predict(self, task_description: str, frames: list, g1_state: dict, step_progress: float):
        return scripted_sorting_chunk(task_description, step_progress, 40, CYCLE_STEPS).tolist()


def disable_flash_attn_on_pre_ampere():
    """flash-attn 2 chỉ chạy trên GPU từ Ampere (sm80) trở lên. Trên T4 (sm75) chặn import flash_attn để
    backbone Qwen3-VL tự chuyển sang attention sdpa. Phải gọi trước khi import gr00t."""
    if HAS_TORCH and torch.cuda.is_available() and torch.cuda.get_device_capability()[0] < 8:
        sys.modules["flash_attn"] = None
        print(
            "[Colab Server] GPU trước Ampere (ví dụ T4): tắt flash-attn, dùng sdpa. bfloat16 trên GPU này chạy "
            "giả lập nên suy luận chậm; nên chọn runtime L4 hoặc A100."
        )


class Gr00tBackend:
    """Model GR00T N1.7 thật qua Gr00tPolicy của Isaac-GR00T."""

    action_space = "real_g1"

    def __init__(self, model_path: str, device: str, embodiment_tag: str = DEFAULT_EMBODIMENT_TAG):
        disable_flash_attn_on_pre_ampere()
        from gr00t.policy.gr00t_policy import Gr00tPolicy

        # Gửi kèm mỗi phản hồi để client kiểm tra đúng checkpoint (đường dẫn gốc, trước khi tải từ HuggingFace)
        self.model_info = {"model_path": model_path, "embodiment_tag": embodiment_tag}
        if not os.path.isdir(model_path):
            from huggingface_hub import snapshot_download
            print(f"[Colab Server] Đang tải checkpoint {model_path} từ HuggingFace...")
            model_path = snapshot_download(model_path)

        print(f"[Colab Server] Đang nạp GR00T từ {model_path} (embodiment {embodiment_tag}) vào {device}...")
        self.policy = Gr00tPolicy(embodiment_tag=embodiment_tag, model_path=model_path, device=device)
        self.embodiment_tag = self.policy.embodiment_tag.value

        cfg = self.policy.get_modality_config()
        self.video_keys = cfg["video"].modality_keys
        self.video_horizon = len(cfg["video"].delta_indices)
        self.state_keys = cfg["state"].modality_keys
        self.language_keys = cfg["language"].modality_keys
        print(f"[Colab Server] Video {self.video_keys} x {self.video_horizon} khung hình | State {self.state_keys}")
        print(f"[Colab Server] Action {cfg['action'].modality_keys}")

    def predict(self, task_description: str, frames: list, g1_state: dict, step_progress: float):
        missing = [k for k in self.state_keys if k not in g1_state]
        if missing:
            raise ValueError(f"Thiếu state {missing}. Client phải gửi g1_state theo embodiment {self.embodiment_tag}.")
        if not frames:
            raise ValueError("Request không có khung hình ego_view.")

        # Thiếu khung hình quá khứ (đầu episode) thì lặp lại khung hình cũ nhất
        frames = list(frames[-self.video_horizon:])
        frames = [frames[0]] * (self.video_horizon - len(frames)) + frames
        video = np.stack(frames)[None].astype(np.uint8)  # (B=1, T, H, W, 3)

        observation = {
            "video": {k: video for k in self.video_keys},
            "state": {k: np.asarray(g1_state[k], dtype=np.float32).reshape(1, 1, -1) for k in self.state_keys},
            "language": {k: [[task_description]] for k in self.language_keys},
        }
        action, _ = self.policy.get_action(observation)
        return {k: v[0].tolist() for k, v in action.items()}  # bỏ batch: {key: [T, D]}

    def warmup(self):
        """Chạy một lần suy luận giả để lần gọi đầu từ client không bị timeout vì khởi tạo CUDA kernel."""
        frame = np.zeros((256, 384, 3), dtype=np.uint8)
        norm_params = self.policy.processor.state_action_processor.norm_params[self.embodiment_tag]["state"]
        state = {}
        for k in self.state_keys:
            dim = norm_params[k]["dim"].item()
            state[k] = np.zeros(dim, dtype=np.float32)
            if k.endswith("eef_9d"):
                state[k][3:] = [1.0, 0.0, 0.0, 0.0, 1.0, 0.0]  # rot6d của ma trận đơn vị
        t0 = time.time()
        self.predict("warm up", [frame], state, 0.0)
        print(f"[Colab Server] Warm-up xong trong {time.time() - t0:.1f}s.")


def decode_frames(payload: dict) -> list:
    """Giải mã khung hình ego_view (JPEG hoặc mảng uint8) thành list ảnh [H, W, 3] theo thứ tự cũ -> mới."""
    frames = payload.get("video", {}).get("ego_view")
    if frames is None:
        return []
    if payload.get("video_encoding") == "jpeg":
        from PIL import Image
        return [np.asarray(Image.open(io.BytesIO(b)).convert("RGB")) for b in frames]
    frames = np.asarray(frames, dtype=np.uint8)
    return list(frames[None] if frames.ndim == 3 else frames)


class Gr00tColabServer:
    """Server suy luận VLA chính sách GR00T-N1.7 qua giao thức ZMQ."""

    def __init__(self, backend, host: str = "0.0.0.0", port: int = 5550):
        if not HAS_ZMQ:
            raise RuntimeError("Thiếu thư viện pyzmq! Vui lòng chạy: pip install pyzmq")
        self.backend = backend
        self.host = host
        self.port = port
        self.context = zmq.Context()
        self.socket = self.context.socket(zmq.REP)
        self.socket.bind(f"tcp://{self.host}:{self.port}")

    def serve(self):
        """Vòng lặp lắng nghe kết nối ZMQ từ máy Local."""
        print("\n" + "=" * 65)
        print(f"🚀 GR00T N1.7 POLICY SERVER ({self.backend.action_space}) ĐANG LẮNG NGHE TẠI: tcp://{self.host}:{self.port}")
        print("   Sẵn sàng nhận ảnh camera và câu lệnh từ máy Local...")
        print("=" * 65 + "\n")

        step_count = 0
        while True:
            try:
                msg_bytes = self.socket.recv()
            except KeyboardInterrupt:
                print("\n[Colab Server] Đang dừng server theo yêu cầu người dùng.")
                break

            # Mọi request đều phải được trả lời (REQ-REP), kể cả khi lỗi
            try:
                payload = pickle.loads(msg_bytes)
                task_desc = payload.get("language", {}).get("annotation.human.task_description", "")
                frames = decode_frames(payload)
                g1_state = payload.get("g1_state", {})
                step_progress = payload.get("step_progress", 0.0)

                step_count += 1
                t0 = time.time()
                action = self.backend.predict(task_desc, frames, g1_state, step_progress)
                dt_ms = (time.time() - t0) * 1000
                img_shape = frames[-1].shape if frames else None
                print(
                    f"[{step_count:04d}] '{task_desc[:40]}' | {len(frames)} khung hình {img_shape} | "
                    f"suy luận {dt_ms:.0f} ms"
                )
                response = {
                    "action": action,
                    "action_space": self.backend.action_space,
                    "model": self.backend.model_info,
                    "inference_time_ms": dt_ms,
                    "step": step_count,
                }
            except Exception as e:
                traceback.print_exc()
                response = {"error": f"{type(e).__name__}: {e}"}
            self.socket.send(pickle.dumps(response))

        self.socket.close()
        self.context.term()
        print("[Colab Server] Đã đóng ZMQ socket an toàn.")


def start_pinggy_tunnel(port: int):
    """Mở đường hầm TCP qua Pinggy (Miễn phí, 0 token, 0 cần đăng ký)."""
    import subprocess
    cmd = [
        "ssh", "-o", "StrictHostKeyChecking=no",
        "-o", "ServerAliveInterval=30",
        "-p", "443",
        f"-R0:localhost:{port}",
        "tcp@free.pinggy.io",
    ]
    try:
        proc = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        print("\n[Pinggy Tunnel] Đang kết nối mạng Pinggy TCP (Không cần tài khoản/token)...")
        pub_url = None
        for _ in range(30):
            line = proc.stdout.readline()
            if not line:
                time.sleep(0.2)
                continue
            if "tcp://" in line:
                parts = line.strip().split("tcp://")
                pub_url = parts[1].split()[0].replace("/", "")
                break

        if pub_url and ":" in pub_url:
            host, remote_port = pub_url.split(":")
            print("\n" + "=" * 65)
            print("🌐 ĐƯỜNG HẦM PINGGY TCP THIẾT LẬP THÀNH CÔNG (KHÔNG CẦN NGROK)!")
            print(f"   Public Address : tcp://{pub_url}")
            print(f"   Host           : {host}")
            print(f"   Port           : {remote_port}")
            print("=" * 65)
            print("👉 LỆNH CHẠY TRÊN MÁY LOCAL CỦA BẠN:")
            print(f"   .venv/bin/python launch_vla_simulation.py \\")
            print(f"       --policy-mode remote \\")
            print(f"       --server-host {host} \\")
            print(f"       --server-port {remote_port}")
            print("=" * 65 + "\n")
        else:
            print("[Pinggy Tunnel] Đang duy trì SSH Pinggy tunnel ở chế độ nền.")
    except Exception as e:
        print(f"[Cảnh báo Pinggy] Không thể mở Pinggy tunnel: {e}")


def main():
    parser = argparse.ArgumentParser(description="Chạy GR00T N1.7 Policy Server trên Colab.")
    parser.add_argument("--host", type=str, default="0.0.0.0", help="Địa chỉ host")
    parser.add_argument("--port", type=int, default=5550, help="Cổng TCP ZMQ")
    parser.add_argument(
        "--policy",
        type=str,
        default="gr00t",
        choices=["gr00t", "scripted"],
        help="gr00t: model GR00T thật (REAL_G1); scripted: quỹ đạo mẫu dạng motion token",
    )
    parser.add_argument("--model-path", type=str, default="nvidia/GR00T-N1.7-3B", help="Model repo hoặc path")
    parser.add_argument(
        "--embodiment-tag",
        type=str,
        default=DEFAULT_EMBODIMENT_TAG,
        help="Embodiment của checkpoint: mặc định REAL_G1 cho checkpoint base; new_embodiment cho checkpoint fine-tune",
    )
    parser.add_argument(
        "--device",
        type=str,
        default="cuda:0" if HAS_TORCH and torch.cuda.is_available() else "cpu",
        help="Thiết bị chạy model",
    )
    parser.add_argument("--ngrok-token", type=str, default=None, help="Ngrok authtoken để mở tunnel")
    parser.add_argument("--pinggy", action="store_true", help="Sử dụng Pinggy TCP tunnel (không cần tài khoản/token)")
    args = parser.parse_args()

    # 1. Mở Pinggy tunnel nếu được yêu cầu (hoặc không có ngrok token)
    if args.pinggy:
        start_pinggy_tunnel(args.port)
    # 2. Mở Ngrok tunnel nếu có token
    elif args.ngrok_token or os.environ.get("NGROK_AUTHTOKEN"):
        try:
            from pyngrok import ngrok
            token = args.ngrok_token or os.environ.get("NGROK_AUTHTOKEN")
            ngrok.set_auth_token(token)
            tunnel = ngrok.connect(args.port, "tcp")
            pub_url = tunnel.public_url.replace("tcp://", "")
            host, port = pub_url.split(":")
            print("\n" + "=" * 65)
            print("🌐 ĐƯỜNG HẦM NGROK TCP ĐÃ THIẾT LẬP THÀNH CÔNG!")
            print(f"   Public Address: {pub_url}")
            print("=" * 65)
            print("👉 LỆNH CHẠY TRÊN MÁY LOCAL CỦA BẠN:")
            print(f"   .venv/bin/python launch_vla_simulation.py \\")
            print(f"       --policy-mode remote \\")
            print(f"       --server-host {host} \\")
            print(f"       --server-port {port}")
            print("=" * 65 + "\n")
        except Exception as e:
            print(f"[Cảnh báo Ngrok] Không thể mở ngrok tunnel: {e}")

    # Nạp model trước khi mở cổng: lỗi nạp model dừng server ngay, không âm thầm chuyển sang quỹ đạo mẫu
    if args.policy == "gr00t":
        backend = Gr00tBackend(args.model_path, args.device, args.embodiment_tag)
        backend.warmup()
    else:
        backend = ScriptedBackend()

    server = Gr00tColabServer(backend, host=args.host, port=args.port)
    server.serve()


if __name__ == "__main__":
    main()

In [ ]:
# [BƯỚC 5] Khởi chạy PolicyServer chạy nền (port 5550), log ghi ra /content/server.log.
# Cell chờ tới khi server sẵn sàng. Bấm ■ chỉ dừng việc chờ, server vẫn chạy; xem trạng thái ở Bước 7, dừng ở Bước 8.
# Checkpoint fine-tune: đường dẫn checkpoint-<bước> trên Drive (Bước 10 của colab_gr00t_finetune.ipynb),
#   thư mục con = DATASET_NAME lúc fine-tune (checkpoint pilot cũ: .../g1_can_sort_pilot/checkpoint-5000).
# Checkpoint base (chỉ để thử, không gắp được lon): lần đầu tải GR00T-N1.7-3B (~7 GB) và backbone Cosmos-Reason2-2B
#   MODEL_PATH = "nvidia/GR00T-N1.7-3B"
#   EMBODIMENT_TAG = "real_g1_relative_eef_relative_joints"
#   (tools/eval_closed_loop.py mặc định dừng nếu server không chạy checkpoint new_embodiment)
# Checkpoint trên Drive được chép về /content/ckpt trước: nạp thẳng từ Drive rất chậm (~13 phút cho 3 shard),
# nạp từ đĩa local nhanh hơn nhiều. Lần chạy sau trên cùng runtime bỏ qua các file đã chép đủ.
# Lần chép đầu vẫn bị giới hạn bởi tốc độ Drive (~15 MB/s, có thể vài phút mỗi shard); tiến độ in mỗi 30 s.
# Nếu Drive rớt (Errno 107) thì tự mount lại và chép tiếp từ chỗ dừng; chạy lại cell cũng chép tiếp từ file .part.
# POLICY = "scripted" để phục vụ quỹ đạo mẫu thay cho model.
MODEL_PATH = "/content/drive/MyDrive/gr00t/checkpoints/g1_can_sort_ik/checkpoint-10000"
EMBODIMENT_TAG = "new_embodiment"
POLICY = "gr00t"

import os, re, socket, subprocess, time
assert os.path.exists("/content/Isaac-GR00T/.venv/bin/python"), (
    "Chưa có môi trường Isaac-GR00T: Bước 2 chưa chạy xong. Chạy lại Bước 2 và xem dòng lỗi đầu tiên."
)
assert os.environ.get("HF_TOKEN"), "Chưa có HF_TOKEN trong phiên này: chạy lại Bước 3 (sau mỗi lần runtime khởi động lại)."


def port_open(port=5550):
    with socket.socket() as s:
        s.settimeout(1)
        return s.connect_ex(("127.0.0.1", port)) == 0


assert not port_open(), "Port 5550 đang có server chạy. Dừng bằng Bước 8 rồi chạy lại cell này."

if MODEL_PATH.startswith("/content/drive"):
    import errno
    from google.colab import drive
    drive.mount("/content/drive")
    assert os.path.isdir(MODEL_PATH), f"Không thấy checkpoint {MODEL_PATH}"
    local = "/content/ckpt/" + "_".join(MODEL_PATH.rstrip("/").split("/")[-2:])
    jobs = [(os.path.join(root, name), os.path.getsize(os.path.join(root, name)))
            for root, _, files in os.walk(MODEL_PATH) for name in sorted(files)]
    total = sum(size for _, size in jobs)
    print(f"Checkpoint trên Drive: {len(jobs)} file, {total / 1e9:.2f} GB")

    def copy_resumable(src, dst, size, chunk=64 << 20):
        # Chép theo khối và nối tiếp vào .part nếu lần trước bị ngắt; in tiến độ mỗi 30 s
        part = dst + ".part"
        done = os.path.getsize(part) if os.path.exists(part) else 0
        t0, t_print, start = time.time(), time.time(), done
        with open(src, "rb") as fi, open(part, "ab") as fo:
            fi.seek(done)
            while done < size:
                buf = fi.read(chunk)
                if not buf:
                    raise OSError(errno.EIO, f"Đọc thiếu {src}: {done}/{size} byte")
                fo.write(buf)
                done += len(buf)
                if time.time() - t_print > 30:
                    rate = (done - start) / 1e6 / (time.time() - t0)
                    print(f"    {os.path.basename(src)}: {done / 1e9:.2f}/{size / 1e9:.2f} GB, {rate:.0f} MB/s, "
                          f"còn khoảng {(size - done) / 1e6 / max(rate, 1e-3) / 60:.1f} phút")
                    t_print = time.time()
        os.replace(part, dst)
        return (done - start) / 1e6 / max(time.time() - t0, 1e-3)

    for src, size in jobs:
        dst = os.path.join(local, os.path.relpath(src, MODEL_PATH))
        if os.path.exists(dst) and os.path.getsize(dst) == size:
            continue
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        if size > 100e6:
            print(f"  chép {os.path.relpath(src, MODEL_PATH)} ({size / 1e9:.2f} GB)...")
        for attempt in range(1, 6):
            try:
                rate = copy_resumable(src, dst, size)
                break
            except OSError as e:
                # Errno 107 (Transport endpoint is not connected): FUSE của Drive bị rớt khi đọc file lớn.
                # Mount lại rồi chép tiếp từ chỗ dừng.
                if e.errno not in (errno.ENOTCONN, errno.EIO) or attempt == 5:
                    raise
                print(f"  ⚠️ {e}; mount lại Drive và chép tiếp (lần {attempt + 1}/5)")
                time.sleep(5)
                drive.mount("/content/drive", force_remount=True)
        if size > 100e6:
            print(f"  xong {os.path.relpath(src, MODEL_PATH)}, {rate:.0f} MB/s")
    print(f"Checkpoint local: {local}")
    MODEL_PATH = local

SERVER_LOG = "/content/server.log"
server_proc = subprocess.Popen(
    ["/content/Isaac-GR00T/.venv/bin/python", "/content/colab_gr00t_server.py", "--port", "5550",
     "--policy", POLICY, "--model-path", MODEL_PATH, "--embodiment-tag", EMBODIMENT_TAG],
    cwd="/content/Isaac-GR00T", stdout=open(SERVER_LOG, "w"), stderr=subprocess.STDOUT,
    env={**os.environ, "PYTHONUNBUFFERED": "1"}, start_new_session=True)
print(f"Model: {MODEL_PATH} | embodiment {EMBODIMENT_TAG} | policy {POLICY}")
print(f"Đã khởi động server (PID {server_proc.pid}). Log: {SERVER_LOG}. Đang chờ server sẵn sàng...")

t0, last = time.time(), ""
while True:
    log = open(SERVER_LOG, errors="replace").read()
    if "ĐANG LẮNG NGHE" in log:
        warmup = re.search(r"Warm-up xong trong [\d.]+s", log)
        print(f"✅ Server sẵn sàng sau {time.time() - t0:.0f} s" + (f" ({warmup.group(0)})." if warmup else "."))
        print("Mở tunnel ở Bước 6 (nếu tunnel cũ còn hạn thì dùng tiếp host/port cũ).")
        break
    if server_proc.poll() is not None:
        print(log[-4000:])
        raise RuntimeError(f"Server đã thoát (mã {server_proc.returncode}); xem log ở trên.")
    tail = re.split(r"[\r\n]+", log.strip())[-1][:160] if log.strip() else ""
    if tail and tail != last:
        print(f"[{time.time() - t0:4.0f} s] {tail}")
        last = tail
    time.sleep(15)

In [ ]:
# [BƯỚC 6] Mở đường hầm TCP Pinggy qua cổng 443 (không cần tài khoản).
# Tunnel miễn phí hết hạn sau khoảng 60 phút: chạy lại cell này để lấy host/port mới, server không cần khởi động lại.
import socket, subprocess, time

if "tunnel_proc" in globals() and tunnel_proc.poll() is None:
    tunnel_proc.terminate()
    print("Đã đóng tunnel cũ.")
with socket.socket() as s:
    s.settimeout(1)
    if s.connect_ex(("127.0.0.1", 5550)) != 0:
        print("⚠️ Chưa có server ở port 5550 (Bước 5 chưa chạy xong). Tunnel vẫn mở được, nhưng local sẽ chưa kết nối được.")

cmd = ["ssh", "-o", "StrictHostKeyChecking=no", "-o", "ServerAliveInterval=30", "-p", "443", "-R0:localhost:5550", "tcp@free.pinggy.io"]
tunnel_proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

print("Đang mở đường hầm TCP qua Pinggy cổng 443...")
time.sleep(3)
pub_url = None
for _ in range(30):
    line = tunnel_proc.stdout.readline()
    if not line:
        time.sleep(0.2)
        continue
    if "tcp://" in line:
        pub_url = line.strip().split("tcp://")[1].split()[0].replace("/", "")
        break

if pub_url and ":" in pub_url:
    host, port = pub_url.split(":")
    print("=" * 75)
    print(f"🎉 Đường hầm TCP: tcp://{pub_url} (mở lúc {time.strftime('%H:%M')}, hết hạn khoảng {time.strftime('%H:%M', time.localtime(time.time() + 3600))})")
    print("=" * 75)
    print("👉 ĐÁNH GIÁ CLOSED-LOOP (terminal máy local):")
    print(f".venv/bin/python tools/eval_closed_loop.py --host {host} --port {port} --dataset datasets/g1_can_sort_bal_test --out eval/<tên> --seconds 20 --stop-on-success")
    print("👉 XEM TRỰC QUAN TRONG GUI:")
    print(f".venv/bin/python launch_vla_simulation.py --policy-mode remote --server-host {host} --server-port {port} --server-timeout-ms 10000")
    print("   Với checkpoint fine-tune từ dataset collect_demos.py (ghi ở 25 Hz), thêm: --vla-action-hz 25")
    print("=" * 75)
else:
    print("Chưa đọc được địa chỉ tunnel; tunnel vẫn chạy nền. Chạy lại cell này nếu cần.")

In [ ]:
# [BƯỚC 7] Trạng thái server: chạy lại bất cứ lúc nào để xem số request, thời gian suy luận và cuối log.
import os, re, socket
log = open("/content/server.log", errors="replace").read() if os.path.exists("/content/server.log") else ""
with socket.socket() as s:
    s.settimeout(1)
    listening = s.connect_ex(("127.0.0.1", 5550)) == 0
print("Server:", "đang lắng nghe ở port 5550" if listening else "KHÔNG chạy")
loaded = re.search(r"Đang nạp GR00T từ (.*?) \(embodiment (\S+)\)", log)
if loaded:
    print(f"Model: {loaded.group(1)} | embodiment {loaded.group(2)}")
reqs = re.findall(r"^\[(\d+)\] .*suy luận (\d+) ms", log, re.M)
if reqs:
    ms = [int(m) for _, m in reqs[-50:]]
    print(f"Đã phục vụ {reqs[-1][0]} request; suy luận TB {sum(ms) / len(ms):.0f} ms (50 request gần nhất).")
print(f"Lỗi (Traceback) trong log: {log.count('Traceback')}")
print("--- cuối log ---")
print("\n".join(re.split(r"[\r\n]+", log.strip())[-15:]))

In [ ]:
# [BƯỚC 8] Dừng server (ví dụ để đổi checkpoint ở Bước 5). Tunnel giữ nguyên.
# Phải đổi STOP_SERVER = True mới dừng, để "Chạy tất cả" không tắt server vừa khởi động ở Bước 5.
STOP_SERVER = False

import subprocess, time
if not STOP_SERVER:
    print("Không dừng server. Muốn dừng thì đặt STOP_SERVER = True rồi chạy lại cell này.")
else:
    subprocess.run(["pkill", "-f", "colab_gr00t_server.py"])
    time.sleep(3)
    print("Đã dừng server." if subprocess.run(["pgrep", "-f", "colab_gr00t_server.py"], capture_output=True).returncode
          else "⚠️ Server vẫn còn chạy; chạy lại cell này.")